# Parte 11 - Dashboard Exploratorio en Python

Generamos visualizaciones exploratorias conectándonos directamente a BigQuery y usando `pandas` + `matplotlib`.

## Configuración

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# Ruta a tu archivo de credenciales
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] =     "../credentials/retail-analytics-lab-grupo2-a4bfa003951d.json"

client = bigquery.Client()
print(f'Proyecto activo: {client.project}')

# Estilo global de los gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

DefaultCredentialsError: File ../credentials/retail-analytics-lab-grupo2-a4bfa00....json was not found.

---
## Gráfico 1 - Ventas por País

In [2]:
sql_pais = """
SELECT
    c.pais,
    ROUND(SUM(v.monto), 2) AS total_ventas
FROM retail_dw.ventas v
JOIN retail_dw.clientes c
    ON v.id_cliente = c.id_cliente
GROUP BY c.pais
ORDER BY total_ventas DESC;
"""

df_pais = client.query(sql_pais).to_dataframe()
print(df_pais)

fig, ax = plt.subplots()
bars = ax.barh(
    df_pais['pais'],
    df_pais['total_ventas'],
    color='steelblue',
    edgecolor='white'
)
ax.set_xlabel('Total Ventas (USD)')
ax.set_title('Ventas Totales por País', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.invert_yaxis()  # El mayor queda arriba

# Etiquetas al final de cada barra
for bar in bars:
    ax.text(
        bar.get_width() * 1.005,
        bar.get_y() + bar.get_height() / 2,
        f"${bar.get_width():,.0f}",
        va='center', fontsize=9
    )

plt.tight_layout()
plt.savefig('ventas_por_pais.png', dpi=150)
plt.show()

NameError: name 'client' is not defined

---
## Gráfico 2 - Top 10 Productos

In [ ]:
sql_productos = """
SELECT
    p.producto,
    ROUND(SUM(v.monto), 2) AS total_ventas
FROM retail_dw.ventas v
JOIN retail_dw.productos p
    ON v.id_producto = p.id_producto
GROUP BY p.producto
ORDER BY total_ventas DESC
LIMIT 10;
"""

df_prod = client.query(sql_productos).to_dataframe()
print(df_prod)

colores = plt.cm.Blues_r([i / len(df_prod) for i in range(len(df_prod))])

fig, ax = plt.subplots()
bars = ax.barh(
    df_prod['producto'],
    df_prod['total_ventas'],
    color=colores,
    edgecolor='white'
)
ax.set_xlabel('Total Ventas (USD)')
ax.set_title('Top 10 Productos por Ventas', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.invert_yaxis()

for bar in bars:
    ax.text(
        bar.get_width() * 1.005,
        bar.get_y() + bar.get_height() / 2,
        f"${bar.get_width():,.0f}",
        va='center', fontsize=9
    )

plt.tight_layout()
plt.savefig('top_productos.png', dpi=150)
plt.show()

---
## Gráfico 3 - Ventas por Sucursal

In [ ]:
sql_sucursal = """
SELECT
    sucursal,
    ROUND(SUM(monto), 2) AS total_ventas
FROM retail_dw.ventas
GROUP BY sucursal
ORDER BY total_ventas DESC;
"""

df_suc = client.query(sql_sucursal).to_dataframe()
print(df_suc)

fig, ax = plt.subplots()
wedges, texts, autotexts = ax.pie(
    df_suc['total_ventas'],
    labels=df_suc['sucursal'],
    autopct='%1.1f%%',
    startangle=140,
    colors=plt.cm.Set3.colors[:len(df_suc)]
)
for at in autotexts:
    at.set_fontsize(9)

ax.set_title('Distribución de Ventas por Sucursal', fontweight='bold')
plt.tight_layout()
plt.savefig('ventas_por_sucursal.png', dpi=150)
plt.show()

---
## Gráfico 4 - Tendencia Mensual de Ventas

In [ ]:
sql_mensual = """
SELECT
    EXTRACT(YEAR  FROM fecha_venta) AS anio,
    EXTRACT(MONTH FROM fecha_venta) AS mes,
    ROUND(SUM(monto), 2)            AS total_ventas
FROM retail_dw.ventas
GROUP BY anio, mes
ORDER BY anio, mes;
"""

df_mes = client.query(sql_mensual).to_dataframe()

# Crear columna de fecha para el eje X
df_mes['periodo'] = pd.to_datetime(
    df_mes['anio'].astype(str) + '-' + df_mes['mes'].astype(str).str.zfill(2)
)
df_mes = df_mes.sort_values('periodo')

print(df_mes[['periodo', 'total_ventas']])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    df_mes['periodo'],
    df_mes['total_ventas'],
    marker='o', linewidth=2,
    color='steelblue', markersize=5
)
ax.fill_between(df_mes['periodo'], df_mes['total_ventas'], alpha=0.15, color='steelblue')
ax.set_xlabel('Mes')
ax.set_ylabel('Total Ventas (USD)')
ax.set_title('Tendencia Mensual de Ventas', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('tendencia_mensual.png', dpi=150)
plt.show()

---
## Resumen de Insights

In [ ]:
print('=' * 55)
print('        RESUMEN EJECUTIVO - RETAIL DW')
print('=' * 55)

print(f"\n🌎 País con más ventas:      {df_pais.iloc[0]['pais']} "
      f"(${df_pais.iloc[0]['total_ventas']:,.2f})")

print(f"📦 Producto estrella:        {df_prod.iloc[0]['producto']} "
      f"(${df_prod.iloc[0]['total_ventas']:,.2f})")

print(f"🏪 Sucursal líder:           {df_suc.iloc[0]['sucursal']} "
      f"(${df_suc.iloc[0]['total_ventas']:,.2f})")

mejor_mes = df_mes.loc[df_mes['total_ventas'].idxmax()]
print(f"📅 Mes con más ventas:       {mejor_mes['periodo'].strftime('%B %Y')} "
      f"(${mejor_mes['total_ventas']:,.2f})")

print('=' * 55)